## Multi-Agentic Systems

## Building Multi-Agentic Systems with Handoff

In [ ]:
from dotenv import load_dotenv
import os
from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, function_tool, handoff
load_dotenv(override=True)
os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "1"

os.environ["OPENAI_API_KEY"] = OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
os.environ["OPENAI_BASE_URL"] = "https://openrouter.ai/api/v1"


In [1]:
@function_tool
def search_documents(query: str) -> str:
    """Search uploaded documents."""
    return f"Document results for '{query}': ..."

billing_agent = Agent(
    name="BillingSpecialist",
    instructions="""You handle billing questions:
    - Payment issues
    - Refunds
    - Invoice requests
    Be helpful and resolve issues quickly.""",
    tools=[search_documents]
)

technical_agent = Agent(
    name="TechnicalSupport",
    instructions="""You handle technical issues:
    - Bug reports
    - How-to questions
    - Feature requests
    Ask clarifying questions to understand the issue."""
)

triage_agent = Agent(
    name="Triage",
    instructions="""You are the first point of contact. Route customers immediately:
- Billing/payment/charge issues → transfer to BillingSpecialist RIGHT AWAY
- Technical problems → transfer to TechnicalSupport RIGHT AWAY

Do NOT ask clarifying questions if the intent is already clear. Transfer immediately.""",
    model="openai/gpt-oss-120b",
    handoffs=[
        handoff(billing_agent),
        handoff(technical_agent)
    ]
)


result = await Runner.run(triage_agent, "I want to know how to fix my machine")
print(result.final_output)

NameError: name 'function_tool' is not defined